# Notebook 03: AVM Model Training

**Goal**: Train a LightGBM model to predict residential sale prices in
Cook County, replicating the CCAO approach in Python.

**Models**:
1. Linear Regression baseline
2. XGBoost baseline
3. LightGBM (primary — matches CCAO's choice)

**Evaluation**: RMSE, MAE, MAPE, R², plus assessor-specific metrics (COD, PRD)

In [6]:
import pandas as pd
import numpy as np
import json
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from time import time
from sklearn.linear_model import Ridge
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score, 
    mean_absolute_percentage_error,
)
import lightgbm as lgb
import xgboost as xgb
sns.set_theme(style="whitegrid")

In [7]:
# Load processed data from Notebook 02
df_train = pd.read_parquet("../data/processed/train_split.parquet")
df_val = pd.read_parquet("../data/processed/val_split.parquet")
df_test = pd.read_parquet("../data/processed/test_split.parquet")

with open("../data/processed/feature_config.json") as f:
    config = json.load(f)

TARGET = config["target_col"]
FEATURES = config["train_features"]
NUMERIC_FEATURES = [c for c in config["all_numeric"] if c in FEATURES]
CAT_FEATURES = [c for c in config["all_categorical"] if c in FEATURES]
TIME_FEATS = [c for c in config["time_features"] if c in FEATURES]
NUMERIC_FEATURES += TIME_FEATS  # Time features are numeric

print(f"Target: {TARGET}")
print(f"Features: {len(FEATURES)} ({len(NUMERIC_FEATURES)} numeric, {len(CAT_FEATURES)} cat)")
print(f"Train: {len(df_train):,}, Val: {len(df_val):,}, Test: {len(df_test):,}")

Target: meta_sale_price
Features: 119 (100 numeric, 19 cat)
Train: 324,088, Val: 40,511, Test: 40,511


In [8]:
# Prepare X, y
X_train = df_train[FEATURES].copy()
y_train = df_train[TARGET].copy()
X_val = df_val[FEATURES].copy()
y_val = df_val[TARGET].copy()
X_test = df_test[FEATURES].copy()
y_test = df_test[TARGET].copy()

# Some columns listed as numeric are actually strings (Arrow string dtype).
# Move any non-numeric "numeric" features into CAT_FEATURES.
misclassified = [c for c in NUMERIC_FEATURES if not pd.api.types.is_numeric_dtype(X_train[c])]
if misclassified:
    print(f"Reclassifying {len(misclassified)} non-numeric columns as categorical: {misclassified}")
    NUMERIC_FEATURES = [c for c in NUMERIC_FEATURES if c not in misclassified]
    CAT_FEATURES = CAT_FEATURES + misclassified

# Ensure categoricals are category dtype for LightGBM
for col in CAT_FEATURES:
    for df in [X_train, X_val, X_test]:
        if col in df.columns:
            df[col] = df[col].astype("category")

print(f"Final: {len(NUMERIC_FEATURES)} numeric, {len(CAT_FEATURES)} categorical")

Reclassifying 4 non-numeric columns as categorical: ['prox_nearest_road_arterial_surface_type', 'prox_nearest_road_collector_surface_type', 'prox_nearest_road_highway_surface_type', 'time_sale_quarter_of_year']
Final: 96 numeric, 23 categorical


## 1. Evaluation Metrics

We use standard ML metrics plus assessor-specific ratio study metrics.

In [9]:
def compute_metrics(y_true, y_pred, label=""):
    """Compute standard ML + assessor-specific metrics."""
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    
    # Filter out zeros/NaN
    mask = (y_true > 0) & (y_pred > 0) & np.isfinite(y_true) & np.isfinite(y_pred)
    y_t, y_p = y_true[mask], y_pred[mask]
    
    # Standard ML metrics
    rmse = np.sqrt(mean_squared_error(y_t, y_p))
    mae = mean_absolute_error(y_t, y_p)
    mape = mean_absolute_percentage_error(y_t, y_p)
    r2 = r2_score(y_t, y_p)
    
    # Assessor-specific: Ratio Study Metrics
    ratios = y_p / y_t  # assessment ratio = predicted / actual
    
    # COD (Coefficient of Dispersion) — measures uniformity
    # Lower is better; IAAO standard: < 15 for residential
    median_ratio = np.median(ratios)
    cod = 100 * np.mean(np.abs(ratios - median_ratio)) / median_ratio
    
    # PRD (Price-Related Differential) — measures regressivity
    # Should be close to 1.0; > 1.03 indicates regressivity
    mean_ratio = np.mean(ratios)
    weighted_mean_ratio = np.sum(y_p) / np.sum(y_t)
    prd = mean_ratio / weighted_mean_ratio
    
    metrics = {
        "RMSE": rmse,
        "MAE": mae,
        "MAPE": mape,
        "R\u00b2": r2,
        "Median Ratio": median_ratio,
        "COD": cod,
        "PRD": prd,
    }
    
    if label:
        print(f"\n=== {label} ===")
        print(f"  RMSE:         ${rmse:>12,.0f}")
        print(f"  MAE:          ${mae:>12,.0f}")
        print(f"  MAPE:         {mape:>12.2%}")
        print(f"  R\u00b2:           {r2:>12.4f}")
        print(f"  Median Ratio: {median_ratio:>12.4f}")
        print(f"  COD:          {cod:>12.2f}  (IAAO target: < 15)")
        print(f"  PRD:          {prd:>12.4f}  (IAAO target: 0.98-1.03)")
    
    return metrics

## 2. Model 1: Ridge Regression Baseline

In [10]:
print("=" * 60)
print("MODEL 1: Ridge Regression Baseline")
print("=" * 60)

# Ridge needs imputation + encoding + scaling
ridge_preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), NUMERIC_FEATURES),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
        ]), CAT_FEATURES),
    ],
    remainder="drop",
)

ridge_model = Pipeline([
    ("preprocessor", ridge_preprocessor),
    ("regressor", Ridge(alpha=1.0)),
])

t0 = time()
ridge_model.fit(X_train, y_train)
ridge_train_time = time() - t0
print(f"Training time: {ridge_train_time:.1f}s")

ridge_pred_val = ridge_model.predict(X_val)
ridge_pred_test = ridge_model.predict(X_test)

ridge_metrics_val = compute_metrics(y_val, ridge_pred_val, "Ridge \u2014 Validation")
ridge_metrics_test = compute_metrics(y_test, ridge_pred_test, "Ridge \u2014 Test")

MODEL 1: Ridge Regression Baseline
Training time: 4.7s

=== Ridge — Validation ===
  RMSE:         $     143,708
  MAE:          $      98,439
  MAPE:               38.38%
  R²:                 0.7297
  Median Ratio:       0.9958
  COD:                 38.54  (IAAO target: < 15)
  PRD:                1.1605  (IAAO target: 0.98-1.03)

=== Ridge — Test ===
  RMSE:         $     154,593
  MAE:          $     107,268
  MAPE:               36.15%
  R²:                 0.7197
  Median Ratio:       0.9171
  COD:                 38.41  (IAAO target: < 15)
  PRD:                1.1544  (IAAO target: 0.98-1.03)


## 3. Model 2: XGBoost Baseline

In [11]:
print("=" * 60)
print("MODEL 2: XGBoost")
print("=" * 60)

# XGBoost needs ordinal-encoded categoricals
xgb_preprocessor = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), NUMERIC_FEATURES),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
        ]), CAT_FEATURES),
    ],
    remainder="drop",
)

# Preprocess
X_train_xgb = xgb_preprocessor.fit_transform(X_train)
X_val_xgb = xgb_preprocessor.transform(X_val)
X_test_xgb = xgb_preprocessor.transform(X_test)

xgb_model = xgb.XGBRegressor(
    n_estimators=1500,
    learning_rate=0.05,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.6,
    min_child_weight=10,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    tree_method="hist",
    early_stopping_rounds=50,
)

t0 = time()
xgb_model.fit(
    X_train_xgb, y_train,
    eval_set=[(X_val_xgb, y_val)],
    verbose=100,
)
xgb_train_time = time() - t0
print(f"Training time: {xgb_train_time:.1f}s")
print(f"Best iteration: {xgb_model.best_iteration}")

xgb_pred_val = xgb_model.predict(X_val_xgb)
xgb_pred_test = xgb_model.predict(X_test_xgb)

xgb_metrics_val = compute_metrics(y_val, xgb_pred_val, "XGBoost \u2014 Validation")
xgb_metrics_test = compute_metrics(y_test, xgb_pred_test, "XGBoost \u2014 Test")

MODEL 2: XGBoost
[0]	validation_0-rmse:271532.00767
[100]	validation_0-rmse:117667.82960
[200]	validation_0-rmse:115360.45322
[300]	validation_0-rmse:114317.84015
[400]	validation_0-rmse:113806.80727
[500]	validation_0-rmse:113491.93724
[600]	validation_0-rmse:113317.89168
[700]	validation_0-rmse:113191.41849
[799]	validation_0-rmse:113186.70221
Training time: 15.7s
Best iteration: 749

=== XGBoost — Validation ===
  RMSE:         $     113,160
  MAE:          $      75,285
  MAPE:               31.67%
  R²:                 0.8325
  Median Ratio:       1.0020
  COD:                 31.61  (IAAO target: < 15)
  PRD:                1.1552  (IAAO target: 0.98-1.03)

=== XGBoost — Test ===
  RMSE:         $     119,688
  MAE:          $      82,194
  MAPE:               30.29%
  R²:                 0.8321
  Median Ratio:       0.9531
  COD:                 31.37  (IAAO target: < 15)
  PRD:                1.1566  (IAAO target: 0.98-1.03)


## 4. Model 3: LightGBM (Primary Model)

This mirrors the CCAO's model choice. LightGBM handles categoricals
natively, which is a major advantage for this data.

In [ ]:
print("=" * 60)
print("MODEL 3: LightGBM ")
print("=" * 60)

# LightGBM Dataset objects — handles categoricals natively
lgb_train = lgb.Dataset(
    X_train[NUMERIC_FEATURES + CAT_FEATURES],
    label=y_train,
    categorical_feature=CAT_FEATURES,
    free_raw_data=False,
)

lgb_val = lgb.Dataset(
    X_val[NUMERIC_FEATURES + CAT_FEATURES],
    label=y_val,
    categorical_feature=CAT_FEATURES,
    reference=lgb_train,
    free_raw_data=False,
)

# Hyperparameters — close to what CCAO uses
# You can tune these with Optuna later for improvement
lgb_params = {
    "objective": "regression",
    "metric": "rmse",
    "boosting_type": "gbdt",
    "learning_rate": 0.05,
    "num_leaves": 512,
    "max_depth": -1,  # No limit; controlled by num_leaves
    "min_data_in_leaf": 50,
    "feature_fraction": 0.6,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "lambda_l1": 0.1,
    "lambda_l2": 1.0,
    "min_gain_to_split": 0.01,
    "max_cat_threshold": 100,
    "cat_smooth": 50,
    "cat_l2": 10,
    "verbose": -1,
    "n_jobs": -1,
    "seed": 42,
}

callbacks = [
    lgb.early_stopping(stopping_rounds=50),
    lgb.log_evaluation(period=100),
]

t0 = time()
lgb_model = lgb.train(
    lgb_params,
    lgb_train,
    num_boost_round=2500,
    valid_sets=[lgb_train, lgb_val],
    valid_names=["train", "valid"],
    callbacks=callbacks,
)
lgb_train_time = time() - t0
print(f"Training time: {lgb_train_time:.1f}s")
print(f"Best iteration: {lgb_model.best_iteration}")

lgb_pred_val = lgb_model.predict(X_val[NUMERIC_FEATURES + CAT_FEATURES])
lgb_pred_test = lgb_model.predict(X_test[NUMERIC_FEATURES + CAT_FEATURES])

lgb_metrics_val = compute_metrics(y_val, lgb_pred_val, "LightGBM \u2014 Validation")
lgb_metrics_test = compute_metrics(y_test, lgb_pred_test, "LightGBM \u2014 Test")

## 5. Model Comparison

In [ ]:
# Compile results
comparison = pd.DataFrame({
    "Ridge (Val)": ridge_metrics_val,
    "Ridge (Test)": ridge_metrics_test,
    "XGBoost (Val)": xgb_metrics_val,
    "XGBoost (Test)": xgb_metrics_test,
    "LightGBM (Val)": lgb_metrics_val,
    "LightGBM (Test)": lgb_metrics_test,
}).T

print("=" * 80)
print("MODEL COMPARISON")
print("=" * 80)
print(comparison.to_string())

In [ ]:
# Visual comparison — test set metrics
test_results = pd.DataFrame({
    "Ridge": ridge_metrics_test,
    "XGBoost": xgb_metrics_test,
    "LightGBM": lgb_metrics_test,
}).T

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

metrics_to_plot = [
    ("RMSE", "lower is better", "steelblue"),
    ("MAE", "lower is better", "coral"),
    ("MAPE", "lower is better", "green"),
    ("R\u00b2", "higher is better", "purple"),
    ("COD", "target < 15", "orange"),
    ("PRD", "target \u2248 1.0", "teal"),
]

for ax, (metric, note, color) in zip(axes.flat, metrics_to_plot):
    vals = test_results[metric]
    bars = ax.bar(vals.index, vals.values, color=color, edgecolor="black", alpha=0.8)
    ax.set_title(f"{metric}\n({note})", fontsize=11)
    ax.set_ylabel(metric)
    
    # Add value labels
    for bar, val in zip(bars, vals.values):
        if metric == "MAPE":
            label = f"{val:.2%}"
        elif metric in ["R\u00b2", "Median Ratio", "PRD"]:
            label = f"{val:.4f}"
        elif metric == "COD":
            label = f"{val:.2f}"
        else:
            label = f"${val:,.0f}"
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                label, ha="center", va="bottom", fontsize=9)
    
    # Add IAAO reference lines
    if metric == "COD":
        ax.axhline(y=15, color="red", linestyle="--", alpha=0.7, label="IAAO max (15)")
        ax.legend(fontsize=8)
    elif metric == "PRD":
        ax.axhline(y=1.03, color="red", linestyle="--", alpha=0.5)
        ax.axhline(y=0.98, color="red", linestyle="--", alpha=0.5)
        ax.set_ylim(0.9, 1.1)

plt.suptitle("AVM Model Comparison \u2014 Test Set Performance", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("../outputs/figures/03_model_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Feature Importance (LightGBM)

In [ ]:
# Built-in feature importance
importance = pd.DataFrame({
    "feature": NUMERIC_FEATURES + CAT_FEATURES,
    "gain": lgb_model.feature_importance(importance_type="gain"),
    "split": lgb_model.feature_importance(importance_type="split"),
})
importance = importance.sort_values("gain", ascending=False)

print("=== Top 30 Features by Gain ===\n")
print(importance.head(30).to_string(index=False))

In [ ]:
# Plot
fig, axes = plt.subplots(1, 2, figsize=(16, 10))

top_n = 25

top_gain = importance.head(top_n)
axes[0].barh(range(top_n), top_gain["gain"].values, color="steelblue", edgecolor="black")
axes[0].set_yticks(range(top_n))
axes[0].set_yticklabels(top_gain["feature"].values, fontsize=8)
axes[0].set_title("Top 25 Features by Gain")
axes[0].invert_yaxis()

top_split = importance.sort_values("split", ascending=False).head(top_n)
axes[1].barh(range(top_n), top_split["split"].values, color="coral", edgecolor="black")
axes[1].set_yticks(range(top_n))
axes[1].set_yticklabels(top_split["feature"].values, fontsize=8)
axes[1].set_title("Top 25 Features by Split Count")
axes[1].invert_yaxis()

plt.suptitle("LightGBM Feature Importance", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("../outputs/figures/03_feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Predicted vs Actual Scatter

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

models = [
    ("Ridge", ridge_pred_test),
    ("XGBoost", xgb_pred_test),
    ("LightGBM", lgb_pred_test),
]

for ax, (name, preds) in zip(axes, models):
    ax.scatter(y_test, preds, alpha=0.05, s=2, color="steelblue")
    
    # Perfect prediction line
    lims = [0, y_test.quantile(0.99)]
    ax.plot(lims, lims, "r--", linewidth=1, label="Perfect prediction")
    
    ax.set_xlim(lims)
    ax.set_ylim(lims)
    ax.set_xlabel("Actual Sale Price ($)")
    ax.set_ylabel("Predicted Sale Price ($)")
    ax.set_title(f"{name}")
    ax.legend()

plt.suptitle("Predicted vs Actual Sale Price (Test Set)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("../outputs/figures/03_pred_vs_actual.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. SHAP Analysis (LightGBM)

SHAP values tell us the per-feature contribution to each prediction.
This is critical for understanding what drives land vs improvement value.

In [ ]:
import shap

print("Computing SHAP values (this may take a few minutes)...")

# Use a sample for SHAP (full dataset is too large)
shap_sample_size = min(5000, len(X_test))
X_shap = X_test[NUMERIC_FEATURES + CAT_FEATURES].sample(
    shap_sample_size, random_state=42
)

explainer = shap.TreeExplainer(lgb_model)
shap_values = explainer.shap_values(X_shap)
print(f"SHAP values computed for {shap_sample_size} samples.")

In [ ]:
# SHAP summary plot
fig, ax = plt.subplots(figsize=(12, 10))
shap.summary_plot(shap_values, X_shap, max_display=25, show=False)
plt.title("SHAP Feature Importance (LightGBM)")
plt.tight_layout()
plt.savefig("../outputs/figures/03_shap_summary.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# SHAP bar plot (mean absolute SHAP value per feature)
fig, ax = plt.subplots(figsize=(10, 8))
shap.summary_plot(shap_values, X_shap, plot_type="bar", max_display=25, show=False)
plt.title("Mean |SHAP Value| by Feature")
plt.tight_layout()
plt.savefig("../outputs/figures/03_shap_bar.png", dpi=150, bbox_inches="tight")
plt.show()

## 9. Save Model & Predictions

In [ ]:
# Save LightGBM model (our primary model)
lgb_model.save_model("../outputs/models/lgb_avm_model.txt")
print("Saved: ../outputs/models/lgb_avm_model.txt")

# Save XGBoost model
joblib.dump(xgb_model, "../outputs/models/xgb_avm_model.joblib")
joblib.dump(xgb_preprocessor, "../outputs/models/xgb_preprocessor.joblib")
print("Saved: ../outputs/models/xgb_avm_model.joblib")

# Save Ridge model
joblib.dump(ridge_model, "../outputs/models/ridge_avm_model.joblib")
print("Saved: ../outputs/models/ridge_avm_model.joblib")

# Save test predictions for later analysis
test_predictions = df_test[["meta_pin", TARGET]].copy()
test_predictions["pred_ridge"] = ridge_pred_test
test_predictions["pred_xgb"] = xgb_pred_test
test_predictions["pred_lgb"] = lgb_pred_test
test_predictions.to_parquet("../outputs/models/test_predictions.parquet", index=False)
print("Saved: ../outputs/models/test_predictions.parquet")

# Save model comparison
comparison.to_csv("../outputs/models/model_comparison.csv")
print("Saved: ../outputs/models/model_comparison.csv")

# Save feature importance
importance.to_csv("../outputs/models/feature_importance.csv", index=False)
print("Saved: ../outputs/models/feature_importance.csv")